# 002 Prefecture Batch Network Check

東京都・福岡県・香川県を対象に、OpenStreetMap の道路ネットワーク取得、可視化、最短経路計算を県ごとに保存しながら実行するための notebook です。

この notebook は以下を前提にしています。
- 県ごとに出力を分離して保存する
- 完了済み県はスキップして途中再開できる
- 失敗県だけを再実行できる

In [10]:
from __future__ import annotations

import json
import random
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox

In [11]:
NOTEBOOK_NAME = "002_prefecture_batch_network_check"
PREFECTURES = [
    "Tokyo, Japan",
    "Fukuoka, Japan",
    "Kagawa, Japan",
]
SEED = 42

# True の場合は success 済みの県をスキップします。
SKIP_COMPLETED = True

# True の場合は failed の県だけを対象にします。
RETRY_FAILED_ONLY = False

# 画面に matplotlib を表示したい場合は True にします。
SHOW_PLOTS = False

# 東京駅から東京スカイツリーまでの追加検証を行う場合は True にします。
RUN_TOKYO_POINT_TO_POINT = True
TOKYO_ROUTE_ORIGIN = "Tokyo Station, Tokyo, Japan"
TOKYO_ROUTE_DESTINATION = "Tokyo Skytree, Tokyo, Japan"

In [12]:
def slugify(value: str) -> str:
    normalized = value.strip().lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized)
    return normalized.strip("_")


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def largest_strongly_connected_subgraph(graph: nx.MultiDiGraph) -> nx.MultiDiGraph:
    components = nx.strongly_connected_components(graph)
    largest_component_nodes = max(components, key=len)
    return graph.subgraph(largest_component_nodes).copy()


def save_network_plot(graph: nx.MultiDiGraph, output_path: Path) -> None:
    fig, _ = ox.plot_graph(
        graph,
        node_size=0,
        edge_color="dimgray",
        edge_linewidth=0.4,
        bgcolor="white",
        show=False,
        close=False,
    )
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_route_plot(graph: nx.MultiDiGraph, route: list[int], output_path: Path) -> None:
    fig, _ = ox.plot_graph_route(
        graph,
        route,
        route_color="crimson",
        route_linewidth=3,
        node_size=0,
        edge_color="lightgray",
        edge_linewidth=0.4,
        bgcolor="white",
        show=False,
        close=False,
    )
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "outputs").exists():
            return candidate
    raise FileNotFoundError("Repository root not found.")


REPO_ROOT = find_repo_root(Path.cwd())
OUTPUTS_ROOT = REPO_ROOT / "outputs"


def prefecture_directories(place_name: str) -> dict[str, Path]:
    place_slug = slugify(place_name)
    figures_dir = OUTPUTS_ROOT / "figures" / NOTEBOOK_NAME / place_slug
    metrics_dir = OUTPUTS_ROOT / "metrics" / NOTEBOOK_NAME / place_slug
    routes_dir = OUTPUTS_ROOT / "routes" / NOTEBOOK_NAME / place_slug

    for directory in (figures_dir, metrics_dir, routes_dir):
        directory.mkdir(parents=True, exist_ok=True)

    return {
        "figures_dir": figures_dir,
        "metrics_dir": metrics_dir,
        "routes_dir": routes_dir,
        "status_path": metrics_dir / "status.json",
        "metrics_json_path": metrics_dir / "network_metrics.json",
        "metrics_csv_path": metrics_dir / "network_metrics.csv",
        "route_nodes_path": routes_dir / "route_nodes.json",
        "route_geojson_path": routes_dir / "route_edges.geojson",
        "network_image_path": figures_dir / "road_network.png",
        "route_image_path": figures_dir / "shortest_route.png",
    }


def read_status(status_path: Path) -> dict:
    if not status_path.exists():
        return {"status": "not_started"}
    return json.loads(status_path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def should_run_prefecture(status: str) -> bool:
    if RETRY_FAILED_ONLY:
        return status == "failed"
    if SKIP_COMPLETED and status == "success":
        return False
    return True


def geocode_point(query: str) -> tuple[float, float]:
    lat, lon = ox.geocode(query)
    return lat, lon

In [6]:
prefecture_status_summary = []

for place_name in PREFECTURES:
    directories = prefecture_directories(place_name)
    status_payload = read_status(directories["status_path"])
    current_status = status_payload.get("status", "not_started")

    prefecture_status_summary.append(
        {
            "place_name": place_name,
            "status": current_status,
            "metrics_dir": str(directories["metrics_dir"]),
        }
    )

prefecture_status_summary

[{'place_name': 'Tokyo, Japan',
  'status': 'not_started',
  'metrics_dir': '/Users/yamadahayato/Documents/GitHub/vrp-road-network-optimization/outputs/metrics/002_prefecture_batch_network_check/tokyo_japan'},
 {'place_name': 'Fukuoka, Japan',
  'status': 'not_started',
  'metrics_dir': '/Users/yamadahayato/Documents/GitHub/vrp-road-network-optimization/outputs/metrics/002_prefecture_batch_network_check/fukuoka_japan'},
 {'place_name': 'Kagawa, Japan',
  'status': 'not_started',
  'metrics_dir': '/Users/yamadahayato/Documents/GitHub/vrp-road-network-optimization/outputs/metrics/002_prefecture_batch_network_check/kagawa_japan'}]

In [ ]:
run_results = []

for prefecture_index, place_name in enumerate(PREFECTURES, start=1):
    directories = prefecture_directories(place_name)
    status_payload = read_status(directories["status_path"])
    current_status = status_payload.get("status", "not_started")

    if not should_run_prefecture(current_status):
        print(f"[{prefecture_index}/{len(PREFECTURES)}] {place_name}: skip ({current_status})")
        run_results.append({"place_name": place_name, "status": current_status, "action": "skipped"})
        continue

    seed_for_prefecture = SEED + prefecture_index
    random.seed(seed_for_prefecture)
    run_started_at = utc_now_iso()

    write_json(
        directories["status_path"],
        {
            "place_name": place_name,
            "status": "running",
            "started_at": run_started_at,
            "seed": seed_for_prefecture,
        },
    )

    print(f"[{prefecture_index}/{len(PREFECTURES)}] {place_name}: start")

    try:
        graph = ox.graph_from_place(place_name, network_type="drive", simplify=True)
        node_count = graph.number_of_nodes()
        edge_count = graph.number_of_edges()

        save_network_plot(graph, directories["network_image_path"])

        routed_graph = largest_strongly_connected_subgraph(graph)
        routed_nodes = list(routed_graph.nodes)
        if len(routed_nodes) < 2:
            raise RuntimeError("最短経路計算に必要なノード数が不足しています。")

        origin, destination = random.sample(routed_nodes, 2)
        route = nx.shortest_path(
            routed_graph,
            source=origin,
            target=destination,
            weight="length",
        )
        route_length_m = nx.shortest_path_length(
            routed_graph,
            source=origin,
            target=destination,
            weight="length",
        )

        save_route_plot(routed_graph, route, directories["route_image_path"])

        route_edges_gdf = ox.routing.route_to_gdf(routed_graph, route, weight="length")
        route_edges_gdf.to_file(directories["route_geojson_path"], driver="GeoJSON")

        metrics = {
            "place_name": place_name,
            "seed": seed_for_prefecture,
            "node_count": node_count,
            "edge_count": edge_count,
            "origin_node": origin,
            "destination_node": destination,
            "shortest_path_distance_m": route_length_m,
            "route_node_count": len(route),
            "road_network_image": str(directories["network_image_path"]),
            "shortest_route_image": str(directories["route_image_path"]),
            "route_geojson": str(directories["route_geojson_path"]),
        }

        write_json(directories["metrics_json_path"], metrics)
        directories["metrics_csv_path"].write_text(
            "metric,value\n" + "\n".join(f"{key},{value}" for key, value in metrics.items()),
            encoding="utf-8",
        )
        write_json(
            directories["route_nodes_path"],
            {"place_name": place_name, "route_nodes": route},
        )
        write_json(
            directories["status_path"],
            {
                "place_name": place_name,
                "status": "success",
                "started_at": run_started_at,
                "completed_at": utc_now_iso(),
                "seed": seed_for_prefecture,
            },
        )

        run_results.append(
            {
                "place_name": place_name,
                "status": "success",
                "node_count": node_count,
                "edge_count": edge_count,
                "shortest_path_distance_m": route_length_m,
            }
        )

        print(f"[{prefecture_index}/{len(PREFECTURES)}] {place_name}: success")

        if SHOW_PLOTS:
            plt.figure(figsize=(8, 8))
            plt.imshow(plt.imread(directories["network_image_path"]))
            plt.axis("off")
            plt.title(f"Road Network: {place_name}")

            plt.figure(figsize=(8, 8))
            plt.imshow(plt.imread(directories["route_image_path"]))
            plt.axis("off")
            plt.title(f"Shortest Route: {place_name}")
            plt.show()

    except Exception as exc:
        error_trace = traceback.format_exc()
        write_json(
            directories["status_path"],
            {
                "place_name": place_name,
                "status": "failed",
                "started_at": run_started_at,
                "failed_at": utc_now_iso(),
                "seed": seed_for_prefecture,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            },
        )
        (directories["metrics_dir"] / "error_traceback.txt").write_text(error_trace, encoding="utf-8")
        run_results.append(
            {
                "place_name": place_name,
                "status": "failed",
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )
        print(f"[{prefecture_index}/{len(PREFECTURES)}] {place_name}: failed ({type(exc).__name__})")

run_results

[1/3] Tokyo, Japan: start


/opt/anaconda3/lib/python3.12/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 651 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [ ]:
summary_path = OUTPUTS_ROOT / "metrics" / NOTEBOOK_NAME / "batch_run_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(
    json.dumps(
        {
            "notebook_name": NOTEBOOK_NAME,
            "prefectures": PREFECTURES,
            "skip_completed": SKIP_COMPLETED,
            "retry_failed_only": RETRY_FAILED_ONLY,
            "results": run_results,
            "generated_at": utc_now_iso(),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
summary_path

## Tokyo Point To Point Check

東京駅から東京スカイツリーまでの道路距離を追加で確認します。

In [13]:
if RUN_TOKYO_POINT_TO_POINT:
    tokyo_dirs = prefecture_directories("Tokyo, Japan")
    tokyo_graph = ox.graph_from_place("Tokyo, Japan", network_type="drive", simplify=True)

    origin_lat, origin_lon = geocode_point(TOKYO_ROUTE_ORIGIN)
    destination_lat, destination_lon = geocode_point(TOKYO_ROUTE_DESTINATION)

    origin_node = ox.distance.nearest_nodes(tokyo_graph, X=origin_lon, Y=origin_lat)
    destination_node = ox.distance.nearest_nodes(tokyo_graph, X=destination_lon, Y=destination_lat)

    tokyo_route = nx.shortest_path(
        tokyo_graph,
        source=origin_node,
        target=destination_node,
        weight="length",
    )
    tokyo_route_length_m = nx.shortest_path_length(
        tokyo_graph,
        source=origin_node,
        target=destination_node,
        weight="length",
    )

    tokyo_route_image_path = tokyo_dirs["figures_dir"] / "tokyo_station_to_tokyo_skytree_route.png"
    save_route_plot(tokyo_graph, tokyo_route, tokyo_route_image_path)

    tokyo_route_geojson_path = tokyo_dirs["routes_dir"] / "tokyo_station_to_tokyo_skytree_route.geojson"
    tokyo_route_edges_gdf = ox.routing.route_to_gdf(tokyo_graph, tokyo_route, weight="length")
    tokyo_route_edges_gdf.to_file(tokyo_route_geojson_path, driver="GeoJSON")

    tokyo_route_nodes_path = tokyo_dirs["routes_dir"] / "tokyo_station_to_tokyo_skytree_route_nodes.json"
    write_json(
        tokyo_route_nodes_path,
        {
            "origin": TOKYO_ROUTE_ORIGIN,
            "destination": TOKYO_ROUTE_DESTINATION,
            "origin_node": origin_node,
            "destination_node": destination_node,
            "route_nodes": tokyo_route,
        },
    )

    tokyo_route_metrics = {
        "origin": TOKYO_ROUTE_ORIGIN,
        "destination": TOKYO_ROUTE_DESTINATION,
        "origin_lat": origin_lat,
        "origin_lon": origin_lon,
        "destination_lat": destination_lat,
        "destination_lon": destination_lon,
        "origin_node": origin_node,
        "destination_node": destination_node,
        "shortest_path_distance_m": tokyo_route_length_m,
        "route_node_count": len(tokyo_route),
        "route_image": str(tokyo_route_image_path),
        "route_geojson": str(tokyo_route_geojson_path),
    }
    tokyo_route_metrics_json_path = tokyo_dirs["metrics_dir"] / "tokyo_station_to_tokyo_skytree_metrics.json"
    tokyo_route_metrics_csv_path = tokyo_dirs["metrics_dir"] / "tokyo_station_to_tokyo_skytree_metrics.csv"
    write_json(tokyo_route_metrics_json_path, tokyo_route_metrics)
    tokyo_route_metrics_csv_path.write_text(
        "metric,value\n" + "\n".join(f"{key},{value}" for key, value in tokyo_route_metrics.items()),
        encoding="utf-8",
    )

    print(f"Tokyo point-to-point shortest path distance: {tokyo_route_length_m:,.2f} m")
    print(f"Saved route image: {tokyo_route_image_path}")
    print(f"Saved route metrics: {tokyo_route_metrics_json_path}")

/opt/anaconda3/lib/python3.12/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 651 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


Tokyo point-to-point shortest path distance: 5,695.67 m
Saved route image: /Users/yamadahayato/Documents/GitHub/vrp-road-network-optimization/outputs/figures/002_prefecture_batch_network_check/tokyo_japan/tokyo_station_to_tokyo_skytree_route.png
Saved route metrics: /Users/yamadahayato/Documents/GitHub/vrp-road-network-optimization/outputs/metrics/002_prefecture_batch_network_check/tokyo_japan/tokyo_station_to_tokyo_skytree_metrics.json


## Travel Time Check

ここからは、東京駅から東京スカイツリーまでの推定所要時間を計算します。

### Assumptions

OpenStreetMap の道路属性を使い、速度情報が不足する道路は道路種別ベースの推定速度で補完します。

### Output Files

結果は東京都向けの `metrics` と `routes` 配下に追加保存します。

### Run

以下のセルで推定所要時間を計算し、結果を保存します。

In [14]:
if RUN_TOKYO_POINT_TO_POINT:
    tokyo_dirs = prefecture_directories("Tokyo, Japan")
    tokyo_graph_for_time = ox.graph_from_place("Tokyo, Japan", network_type="drive", simplify=True)
    tokyo_graph_for_time = ox.add_edge_speeds(tokyo_graph_for_time)
    tokyo_graph_for_time = ox.add_edge_travel_times(tokyo_graph_for_time)

    origin_lat, origin_lon = geocode_point(TOKYO_ROUTE_ORIGIN)
    destination_lat, destination_lon = geocode_point(TOKYO_ROUTE_DESTINATION)

    origin_node = ox.distance.nearest_nodes(tokyo_graph_for_time, X=origin_lon, Y=origin_lat)
    destination_node = ox.distance.nearest_nodes(tokyo_graph_for_time, X=destination_lon, Y=destination_lat)

    tokyo_fastest_route = nx.shortest_path(
        tokyo_graph_for_time,
        source=origin_node,
        target=destination_node,
        weight="travel_time",
    )
    tokyo_fastest_route_seconds = nx.shortest_path_length(
        tokyo_graph_for_time,
        source=origin_node,
        target=destination_node,
        weight="travel_time",
    )
    tokyo_fastest_route_distance_m = nx.shortest_path_length(
        tokyo_graph_for_time,
        source=origin_node,
        target=destination_node,
        weight="length",
    )

    tokyo_fastest_route_image_path = tokyo_dirs["figures_dir"] / "tokyo_station_to_tokyo_skytree_fastest_route.png"
    save_route_plot(tokyo_graph_for_time, tokyo_fastest_route, tokyo_fastest_route_image_path)

    tokyo_fastest_route_geojson_path = tokyo_dirs["routes_dir"] / "tokyo_station_to_tokyo_skytree_fastest_route.geojson"
    tokyo_fastest_route_edges_gdf = ox.routing.route_to_gdf(tokyo_graph_for_time, tokyo_fastest_route, weight="travel_time")
    tokyo_fastest_route_edges_gdf.to_file(tokyo_fastest_route_geojson_path, driver="GeoJSON")

    tokyo_fastest_route_nodes_path = tokyo_dirs["routes_dir"] / "tokyo_station_to_tokyo_skytree_fastest_route_nodes.json"
    write_json(
        tokyo_fastest_route_nodes_path,
        {
            "origin": TOKYO_ROUTE_ORIGIN,
            "destination": TOKYO_ROUTE_DESTINATION,
            "origin_node": origin_node,
            "destination_node": destination_node,
            "route_nodes": tokyo_fastest_route,
        },
    )

    tokyo_fastest_route_metrics = {
        "origin": TOKYO_ROUTE_ORIGIN,
        "destination": TOKYO_ROUTE_DESTINATION,
        "origin_node": origin_node,
        "destination_node": destination_node,
        "estimated_travel_time_seconds": tokyo_fastest_route_seconds,
        "estimated_travel_time_minutes": tokyo_fastest_route_seconds / 60,
        "route_distance_m": tokyo_fastest_route_distance_m,
        "route_node_count": len(tokyo_fastest_route),
        "route_image": str(tokyo_fastest_route_image_path),
        "route_geojson": str(tokyo_fastest_route_geojson_path),
    }
    tokyo_fastest_route_metrics_json_path = tokyo_dirs["metrics_dir"] / "tokyo_station_to_tokyo_skytree_fastest_route_metrics.json"
    tokyo_fastest_route_metrics_csv_path = tokyo_dirs["metrics_dir"] / "tokyo_station_to_tokyo_skytree_fastest_route_metrics.csv"
    write_json(tokyo_fastest_route_metrics_json_path, tokyo_fastest_route_metrics)
    tokyo_fastest_route_metrics_csv_path.write_text(
        "metric,value\n" + "\n".join(f"{key},{value}" for key, value in tokyo_fastest_route_metrics.items()),
        encoding="utf-8",
    )

    print(f"Estimated travel time: {tokyo_fastest_route_seconds / 60:,.2f} minutes")
    print(f"Route distance: {tokyo_fastest_route_distance_m:,.2f} m")
    print(f"Saved fastest-route metrics: {tokyo_fastest_route_metrics_json_path}")

/opt/anaconda3/lib/python3.12/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 651 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


Estimated travel time: 7.56 minutes
Route distance: 5,695.67 m
Saved fastest-route metrics: /Users/yamadahayato/Documents/GitHub/vrp-road-network-optimization/outputs/metrics/002_prefecture_batch_network_check/tokyo_japan/tokyo_station_to_tokyo_skytree_fastest_route_metrics.json
